Damped Newton method for chi=32 truncating to 15,15

Start Newton method after 4 RG steps.

Damping with newton_step=0.5 is activated a couple of times


In [1]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [2]:
gilt_eps = 2e-5
chi = 32
trunc_shape = [15 15; 15 15; 15 15; 15 15]  # shape to truncate to, not to deal with Gilt tensor dimension oscillations
cg_eps = 1e-10
newton_eps = 1e-9
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 1,
	"rotate" => true
)
Jratio = 1.0


relT=1.0
rg_steps = 6
#do rg_steps steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, rg_steps, gilt_pars)["A"];
#NB traj consists of PyObjects


traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[rg_steps+1], accepted_elements, _ = fix_discrete_gauge(traj[rg_steps+1]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

for i in 1:length(traj)
    println(i," ",traj[i].shape, traj[i].qhape )
end

distances = Float64[]
for i in 4:length(traj)-1
	if i <= 10
		push!(distances, embedded_distance_with_additional_sign_fixing(traj[i], traj[i+1]))
	else
		try
			push!(distances, embedded_distance(traj[i], traj[i+1]))
		catch
			push!(distances, NaN)
		end
	end
end

println(distances)

A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

┌ Warning: new_list_of_elements: new entry is below the threshold. It was 3.0000754996684248e-5 and became 4.531323895714213e-8. Index CartesianIndex(1, 1, 16, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 2.8867589746690045e-6 and became 3.436895514617369e-8. Index CartesianIndex(1, 1, 1, 16) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466


1 

┌ Warning: new_list_of_elements: new entry is below the threshold. It was -1.0910732246514878e-5 and became 5.0134926058782275e-8. Index CartesianIndex(1, 1, 30, 18) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466


[1 1; 1 1; 1 1; 1 1][0 1; 0 1; 0 1; 0 1]
2 [2 2; 2 2; 2 2; 2 2][0 1; 0 1; 0 1; 0 1]
3 [8 8; 8 8; 8 8; 8 8][0 1; 0 1; 0 1; 0 1]
4 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
5 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
6 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
7 [16 16; 16 16; 16 16; 16 16][0 1; 0 1; 0 1; 0 1]
[0.057888078717713835, 0.04217346749090768, 0.02376033736092818]


In [3]:
A[1] = truncate_blocks(traj[4], trunc_shape)
for i in 1:30
    println("i=",i)
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    e0, RAshape = fp_error_with_shape(A[i],accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||R(A[i])-A[i]||= ", e0)
    println("shapes:", A[i].shape, RAshape)
    flush(stdout)
    if A[i].shape != RAshape
        throw(ErrorException("shapes unequal"))
    end
    deltaA[i] = newton_correction_with_iterations_fixed(A[i], 10, accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    newton_step = 1.0
    enew = e0
    while true #damped Newton method implementation, which reduces a step by 2 if cost function does not decrease
        println("newton_step= ", newton_step)
        Anew = A[i] + newton_step * deltaA[i]
        Anew, accepted_elements_new = fix_discrete_gauge(Anew; tol = 1e-7);
        enew, RAnewshape = fp_error_with_shape(Anew, accepted_elements_new, gilt_pars; trunc_shape = trunc_shape)
        println("fp_error= ", enew)
        println("shapes:", Anew.shape, RAnewshape)
        if enew < e0 && Anew.shape == RAnewshape
            A[i+1] = Anew
            break
        end
        newton_step *= 0.5 
    end
    if enew < newton_eps
        break
    end
end

i=1
||R(A[i])-A[i]||= 0.05788717811444853
shapes:[15 15; 15 15; 15 15; 15 15][15 15; 15 15; 15 15; 15 15]
Dict{Any, Any}((1, "N") => 28, (1, "W") => 22, (1, "S") => 25, (1, "E") => 22, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  10 eigenvalues converged
│ *  norm of residuals = (2.2390837041079377e-46, 3.582822661307155e-32, 8.367226603932675e-32, 3.498928148882685e-23, 2.5451097606009357e-22, 6.164442824285359e-19, 6.164442824285359e-19, 2.481653759539917e-14, 2.380068627646039e-13, 2.380068627646039e-13)
└ *  number of operations = 51


EIGENVALUES (INITIAL):
1.985030475147394 + 0.0im
-0.9350608755302774 + 0.0im
-0.9250697325834257 + 0.0im
0.5570672094214961 + 0.0im
0.5512567933578671 + 0.0im
0.0003195576389210343 + 0.4198696084319775im
0.0003195576389210343 - 0.4198696084319775im
-0.3104298699400913 + 0.0im
0.0007154784197545356 + 0.3068089695766854im
0.0007154784197545356 - 0.3068089695766854im
||deltaA[i]||= 0.13386609250927864
newton_step= 1.0


┌ Warning: new_list_of_elements: new entry is below the threshold. It was -7.469890285773619e-5 and became -3.967260190667861e-8. Index CartesianIndex(1, 22, 29, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466


fp_error= 0.027313665087036483
shapes:[15 15; 15 15; 15 15; 15 15][15 15; 15 15; 15 15; 15 15]
i=2


...

┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (6.906143764700125e-53, 1.1569964478404268e-38, 4.906039943494597e-38, 7.972869861053644e-29, 1.3380409669582322e-28, 6.21663090590606e-26, 6.21663090590606e-26, 2.5436940048743003e-18, 3.1927613655451533e-18, 4.239659943493654e-15, 4.239659943493654e-15)
└ *  number of operations = 59


EIGENVALUES (INITIAL):
1.9982246094716332 + 0.0im
-0.9941355238946665 + 0.0im
-0.9864964625964546 + 0.0im
0.6091552868860673 + 0.0im
0.6045548616763736 + 0.0im
-0.007890196464296275 + 0.5060610558943681im
-0.007890196464296275 - 0.5060610558943681im
0.3797280702123904 + 0.0im
-0.3536891009006304 + 0.0im
-0.1474379009932565 + 0.24668359473669904im
-0.1474379009932565 - 0.24668359473669904im
||deltaA[i]||= 0.00031367229937984413
newton_step= 1.0
fp_error= 8.76386016509744e-5
shapes:[15 15; 15 15; 15 15; 15 15][15 15; 15 15; 15 15; 15 15]
i=8
||R(A[i])-A[i]||= 8.76386016509744e-5
shapes:[15 15; 15 15; 15 15; 15 15][15 15; 15 15; 15 15; 15 15]
Dict{Any, Any}((1, "N") => 71, (1, "W") => 48, (1, "S") => 43, (1, "E") => 48, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (1.3610080916457615e-52, 2.8074412083030547e-38, 1.0523832362149847e-37, 1.5886629347340873e-30, 9.22762710244458e-29, 3.134049440181682e-26, 3.134049440181682e-26, 1.3603020492342458e-18, 1.561289406692202e-16, 7.062695687912706e-15, 7.062695687912706e-15)
└ *  number of operations = 58


EIGENVALUES (INITIAL):
1.9983208402324113 + 0.0im
-0.9945995790217212 + 0.0im
-0.9867002230749848 + 0.0im
0.6399001010205688 + 0.0im
0.5970564917394037 + 0.0im
-0.00986435453719599 + 0.5027846488187528im
-0.00986435453719599 - 0.5027846488187528im
0.3732137854809835 + 0.0im
-0.3490712946271422 + 0.0im
-0.14815825999611182 + 0.24553021642951203im
-0.14815825999611182 - 0.24553021642951203im
||deltaA[i]||= 0.00014207288622793062
newton_step= 1.0
fp_error= 2.969155297925238e-5
shapes:[15 15; 15 15; 15 15; 15 15][15 15; 15 15; 15 15; 15 15]
i=9
||R(A[i])-A[i]||= 2.969155297925238e-5
shapes:[15 15; 15 15; 15 15; 15 15][15 15; 15 15; 15 15; 15 15]
Dict{Any, Any}((1, "N") => 71, (1, "W") => 48, (1, "S") => 43, (1, "E") => 48, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


LoadError: InterruptException: